In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# Paths assume the notebook runs from the notebooks folder
data_dir = Path("../data")
image_dir = data_dir / "images"
processed_dir = data_dir / "processed"

# Restore the prepared metadata and exact output-label order
df = pd.read_csv(processed_dir / "metadata_with_splits.csv")

with open(processed_dir / "disease_labels.json") as file:
    disease_labels = json.load(file)

-------------------------------------

## PyTorch Data Pipeline

The model will train on individual chest X-ray images paired with their 14 binary disease labels.

PyTorch typically handles this through two main components:

- A **Dataset** defines how to retrieve one training example at a time. For this project, one example consists of an image and its 14-label target vector.
- A **DataLoader** takes those individual examples and groups them into batches for model training and evaluation.

Before creating these objects, the image preprocessing steps also need to be defined. Each image will eventually be converted into a tensor, resized to the dimensions expected by the neural network, and normalized before being passed to the model.

In [2]:
# Create seperate image-level DataFrames for each dataset partiion

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

len(train_df), len(val_df), len(test_df)

(78487, 16828, 16805)

In [3]:
train_df.head(100)

,image_index,finding_labels,follow_up,patient_id,patient_age,patient_sex,view_position,image_width,image_height,pixel_spacing_x,...,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,num_findings,split
3,00000002_000.png,No Finding,0,2,80,M,PA,2500,2048,0.171,...,0,0,0,1,0,0,0,0,1,train
4,00000003_001.png,Hernia,0,3,74,F,PA,2500,2048,0.168,...,1,0,0,0,0,0,0,0,1,train
5,00000003_002.png,Hernia,1,3,75,F,PA,2048,2500,0.168,...,1,0,0,0,0,0,0,0,1,train
6,00000003_003.png,Hernia|Infiltration,2,3,76,F,PA,2698,2991,0.143,...,1,1,0,0,0,0,0,0,2,train
7,00000003_004.png,Hernia,3,3,77,F,PA,2500,2048,0.168,...,1,0,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,00000032_008.png,No Finding,8,32,54,F,AP,2500,2048,0.168,...,0,0,0,1,0,0,0,0,1,train
123,00000032_009.png,No Finding,9,32,54,F,AP,2500,2048,0.168,...,0,0,0,1,0,0,0,0,1,train
124,00000032_010.png,No Finding,10,32,54,F,AP,2500,2048,0.168,...,0,0,0,1,0,0,0,0,1,train
125,00000032_011.png,Infiltration,11,32,54,F,AP,2500,2048,0.168,...,0,1,0,0,0,0,0,0,1,train


In [4]:
# Look at one training row
train_df.iloc[0]

image_index           00000002_000.png
finding_labels              No Finding
follow_up                            0
patient_id                           2
patient_age                         80
patient_sex                          M
view_position                       PA
image_width                       2500
image_height                      2048
pixel_spacing_x                  0.171
pixel_spacing_y                  0.171
Atelectasis                          0
Cardiomegaly                         0
Consolidation                        0
Edema                                0
Effusion                             0
Emphysema                            0
Fibrosis                             0
Hernia                               0
Infiltration                         0
Mass                                 0
No Finding                           1
Nodule                               0
Pleural_Thickening                   0
Pneumonia                            0
Pneumothorax             

### What One Dataset Example Contains

Each PyTorch training example will contain:

1. the chest X-ray image referenced by `image_index`
2. a 14-element binary target vector containing the disease labels for that image

For example, an image labeled with both Effusion and Atelectasis will have `1` in those two target positions and `0` for the remaining disease labels.

The Dataset class will be responsible for loading the correct image file, applying any preprocessing transforms, and returning the image together with this target vector.

In [5]:
import torch
from torch.utils.data import Dataset

### Custom PyTorch Dataset

The custom `Dataset` class defines how PyTorch retrieves one example from the image-level DataFrame.

Its three required methods are:

- `__init__`: stores the DataFrame, image directory, label columns, and optional transforms
- `__len__`: returns the number of images in the Dataset
- `__getitem__`: loads the requested image, builds its 14-label target tensor, applies preprocessing, and returns the image-label pair

In [6]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, label_columns, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.label_columns = label_columns
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["image_index"]
        image = Image.open(image_path).convert("L")

        labels = torch.tensor(
            row[self.label_columns].to_numpy(dtype=np.float32), dtype=torch.float32
        )

        if self.transform:
            image = self.transform(image)

        return image, labels

In [7]:
# test the new class
train_dataset = ChestXrayDataset(
    dataframe=train_df, image_dir=image_dir, label_columns=disease_labels
)

len(train_dataset)

78487

In [8]:
image, labels = train_dataset[0]

print(type(image))
print(image.size)
print(labels)
print(labels.shape)

<class 'PIL.Image.Image'>
(1024, 1024)
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
torch.Size([14])


The custom Dataset successfully retrieves an individual chest X-ray and its corresponding 14-element disease-label tensor.

The sample image remains a `PIL.Image` at this stage because no image transforms have been applied yet. Its target tensor has shape `[14]`, matching the 14 disease labels used for multi-label classification.

### Image Preprocessing

The downloaded radiographs are 1024 × 1024 grayscale images. For the initial baseline, images will be resized to **224 × 224** to match a common input size used with ImageNet-pretrained convolutional neural networks and to keep training computationally manageable.

This substantial reduction in resolution may remove subtle radiographic detail, which is particularly important in chest X-ray interpretation. Higher-resolution inputs will therefore be considered later as a model improvement and compared against the 224 × 224 baseline.

The images are also converted from one grayscale channel to three identical channels so they are compatible with pretrained RGB networks. This does not add new image information.

In [9]:
from torchvision import transforms

In [10]:
# Prepare grayscale X-rays for pretrained CNN
base_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            # Standard ImageNet R/G/B mean and standard deviation for pretrained torchvision models.
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

In [11]:
# Check sample
sample_image = Image.open(image_dir / train_df.iloc[0]["image_index"]).convert("L")

transformed_image = base_transform(sample_image)

print(type(transformed_image))
print(transformed_image.shape)
print(transformed_image.min())
print(transformed_image.max())

<class 'torch.Tensor'>
torch.Size([3, 224, 224])
tensor(-2.1179)
tensor(2.4657)


The preprocessing transform successfully converted the grayscale X-ray into a normalized PyTorch tensor with shape `[3, 224, 224]`.

The three channels are identical copies of the original grayscale image, while the negative and positive tensor values result from ImageNet normalization. This confirms that the image is now in the format expected by a pretrained convolutional neural network.

In [12]:
# Create train_dataset again using our base_transform
train_dataset = ChestXrayDataset(
    dataframe=train_df,
    image_dir=image_dir,
    label_columns=disease_labels,
    transform=base_transform,
)

In [13]:
# Check sample
image, labels = train_dataset[0]

print(type(image))
print(image.shape)
print(labels.shape)

<class 'torch.Tensor'>
torch.Size([3, 224, 224])
torch.Size([14])


### Create DataLoaders

The `Dataset` retrieves one image-label pair at a time. A PyTorch `DataLoader` groups those individual examples into batches that can be passed through the neural network efficiently.

For training, the data are shuffled between epochs so the model does not repeatedly see images in the same order. Validation and test data are not shuffled because their order does not affect evaluation.

In [14]:
from torch.utils.data import DataLoader

In [15]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [16]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32, 14])


In [17]:
val_dataset = ChestXrayDataset(
    dataframe=val_df,
    image_dir=image_dir,
    label_columns=disease_labels,
    transform=base_transform,
)

test_dataset = ChestXrayDataset(
    dataframe=test_df,
    image_dir=image_dir,
    label_columns=disease_labels,
    transform=base_transform,
)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

len(train_dataset), len(val_dataset), len(test_dataset)

(78487, 16828, 16805)

## Baseline CNN Model

The initial model will use **DenseNet-121** with weights pretrained on ImageNet.

Transfer learning provides a useful starting point because the early layers of a pretrained convolutional neural network have already learned general visual features such as edges, textures, and shapes. These features can then be adapted to chest radiographs rather than training the entire network from randomly initialized weights.

DenseNet-121 is used as the baseline architecture because it is a well-established convolutional network that has been widely applied to chest X-ray classification.

The original DenseNet classifier predicts 1,000 ImageNet classes. For this project, that final layer must be replaced so that the network produces **14 outputs, one for each disease label**.

In [18]:
from torchvision import models

In [19]:
# Load DenseNet-121 with pretrained ImageNet weights
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

model.classifier

Linear(in_features=1024, out_features=1000, bias=True)

In [20]:
# 1. Read how many inputs the OLD layer receives
number_of_inputs = model.classifier.in_features

# 2. Create a NEW layer with the same input size and 14 outputs
new_layer = torch.nn.Linear(
    in_features=number_of_inputs, out_features=len(disease_labels)
)

# 3. Replace the old classifier with the new layer
model.classifier = new_layer

In [21]:
# Take one training batch and pass the images through the model
images, labels = next(iter(train_loader))
outputs = model(images)

# Confirm that the model produces one output for each of the 14 disease labels
print(outputs.shape)

# Confirm that the target tensor has the matching 14-label shape
print(labels.shape)

torch.Size([32, 14])
torch.Size([32, 14])


### Logits and Multi-Label Output

The model currently produces 14 raw output values for each image. These values are called **logits**.

A logit is not yet a probability. It can be any real number, including negative or positive values.

Because this is a **multi-label** problem, each disease is treated independently. An image may be positive for several diseases at the same time, so the outputs should not compete with one another as they would in a multiclass softmax model.

During evaluation, a **sigmoid** function can convert each logit independently into a probability between 0 and 1.

In [22]:
# Inspect the 14 raw logits produced for the first image in the batch
outputs[0]

tensor([-0.1583, -0.4729, -0.5527, -0.2876,  0.2648,  0.4124, -0.5737, -0.0956,
        -0.4834, -0.5423,  0.3012,  0.2211, -0.2418, -0.8751],
       grad_fn=<SelectBackward0>)

In [23]:
# Convert the 14 raw logits for the first image into independent probabilities
probabilities = torch.sigmoid(outputs[0])

probabilities

tensor([0.4605, 0.3839, 0.3652, 0.4286, 0.5658, 0.6017, 0.3604, 0.4761, 0.3814,
        0.3676, 0.5747, 0.5550, 0.4398, 0.2942], grad_fn=<SigmoidBackward0>)

### Loss Function

Because each disease label is a separate binary outcome, the model needs a loss function that can evaluate all 14 labels independently for each image.

`BCEWithLogitsLoss` is appropriate for this multi-label task because it combines two steps:

- binary cross-entropy, which compares each predicted disease score with the true `0` or `1` label
- sigmoid, which converts each raw logit into a probability internally

Using the combined function is numerically more stable than applying sigmoid manually before calculating binary cross-entropy.

In [24]:
criterion = torch.nn.BCEWithLogitsLoss()

In [25]:
# Calculate the loss between the model's raw logits and the true 14-label targets.
loss = criterion(outputs, labels)

loss

tensor(0.6320, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

### Freeze the Pretrained Feature Extractor

For the initial baseline, the pretrained DenseNet feature-extraction layers are frozen so their weights are not updated during training.

Only the newly added 14-output classifier will be trained. This provides a simple transfer-learning baseline and lets us evaluate how useful the pretrained visual features are before fine-tuning deeper layers of the network.

In [26]:
# Freeze all pretrained DenseNet parameters
for parameter in model.features.parameters():
    parameter.require_grad = False

In [27]:
# Update only the new classifier parameters during baseline training
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

### Baseline Training Components

At this point, the baseline training setup includes:

- **Model:** DenseNet-121 adapted to produce 14 disease outputs
- **Loss function:** `BCEWithLogitsLoss` for multi-label classification
- **Trainable parameters:** only the newly added classifier layer
- **Optimizer:** Adam, which updates the classifier weights during training

These components are now ready to be combined in the training loop.

----------------------------------------------------

### Training Loop

Training repeats the same sequence for each batch of images:

1. **Forward pass:** send the image batch through the model to produce 14 logits per image.
2. **Calculate loss:** compare those logits with the true 14-label targets using `BCEWithLogitsLoss`.
3. **Backward pass:** calculate how each trainable parameter contributed to the error.
4. **Optimizer step:** update the classifier weights in the direction that reduces the loss.

This process is repeated across all training batches. One complete pass through the training dataset is called an **epoch**.

In [ ]:
# Define a single epoch
model.train()

running_loss = 0.0

for images, labels in train_loader:
    # Remove gradients left over from the previous batch
    optimizer.zero_grad()

    # Forward pass: generate 14 logits for each image
    outputs = model(images)

    # Compare predictions with the true multi label targets
    loss = criterion(outputs, labels)

    # Calculate gradients for the trainable classifier parameters
    loss.backward()

    #Update the classifier weights
    optimizer.step()

    running_loss += loss.item()

average_train_loss = running_loss / len(train_loader)

average_train_loss

we have 78,487 images on CPU (no GPU in the laptop), so a full DenseNet epoch may be slow even with the feature extractor frozen.

want to decide whether to test the loop on a small subset first before committing to a full epoch.

test the training loop on a small subset first, then run the full epoch once we know nothing is broken.

In [34]:
from torch.utils.data import DataLoader, Subset

# Use a small subset of the training data to confirm the loop works

subset_size = 512

train_subset = Subset(
    train_dataset,
    range(subset_size)
)

test_train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True
)

In [35]:
model.train()

running_loss = 0.0

for images, labels in test_train_loader:
    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

average_train_loss = running_loss / len(test_train_loader)

average_train_loss


0.30785505287349224

### Local Training Check

A small subset of the training data was used to verify that the complete training loop runs successfully on the local CPU.

The test confirmed that images can pass through the DataLoader and model, the multi-label loss can be calculated, gradients can be backpropagated, and the optimizer can update the classifier weights.

Because full DenseNet-121 training on the complete dataset would be inefficient on the local CPU, subsequent model training will be performed using GPU acceleration on Google Colab